<h1 align="center">Data Cleaning</h1>

In [118]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [119]:
if not os.path.isdir('dataset') and os.path.isdir('../dataset'):
    os.chdir('..')

filename = os.path.join('dataset', '311_Service_Requests_from_2020_to_Present.zip')
df = pd.read_csv(filename, low_memory=False)
df.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,70471747,09/20/2026 01:51:01 AM,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,NaN,Street/Sidewalk,10016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.744553,-73.977398,POINT (-73.977397686134 40.744552551425)
1,70474903,09/20/2026 01:50:51 AM,NaN,DEP,Department of Environmental Protection,Water Maintenance,Water Main Break,NaN,Street,11238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.672907,-73.959725,POINT (-73.959725084153 40.672906659623)
2,70470200,09/20/2026 01:50:51 AM,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11373,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.737271,-73.879943,POINT (-73.879942579006 40.737270748449)
3,70475185,09/20/2026 01:50:45 AM,NaN,NYPD,New York City Police Department,Noise - Vehicle,Car/Truck Music,NaN,Street/Sidewalk,11220,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.641947,-73.999272,POINT (-73.999272130506 40.641947092776)
4,70475938,09/20/2026 01:50:40 AM,NaN,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,Street/Sidewalk,11433,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689879,-73.791066,POINT (-73.791066478538 40.689878873276)


In [120]:
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
df['Closed Date'] = pd.to_datetime(df['Closed Date'], errors='coerce')

missing_closed_date = df['Closed Date'].isna().sum()
print(f"Number of missing 'Closed Date' values: {missing_closed_date}")
df = df.dropna(subset=['Closed Date'])

df['resolution_hours'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600

negative_resolution = (df['resolution_hours'] < 0).sum()
print(f"Number of rows with negative resolution time: {negative_resolution}")
df = df[df['resolution_hours'] >= 0]

Number of missing 'Closed Date' values: 179879
Number of rows with negative resolution time: 1011


In [121]:
df.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location,resolution_hours
75,70475296,2026-09-20 01:39:36,2026-09-20 01:47:32,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11208,...,NaN,NaN,NaN,NaN,NaN,NaN,40.666362,-73.876732,POINT (-73.876732316369 40.666361759266),0.132222
90,70471164,2026-09-20 01:37:27,2026-09-20 01:47:26,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11222,...,NaN,NaN,NaN,NaN,NaN,NaN,40.730319,-73.960665,POINT (-73.960664569783 40.73031925345),0.166389
95,70470378,2026-09-20 01:36:25,2026-09-20 01:45:23,NYPD,New York City Police Department,Noise - Park,Loud Music/Party,NaN,Park/Playground,10033,...,NaN,NaN,NaN,NaN,NaN,NaN,40.843501,-73.945906,POINT (-73.945905791727 40.843500551231),0.149444
99,70478136,2026-09-20 01:35:40,2026-09-20 01:49:29,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,NaN,Street/Sidewalk,11238,...,NaN,NaN,NaN,NaN,NaN,NaN,40.676053,-73.961223,POINT (-73.961222929757 40.676052686722),0.230278
106,70478077,2026-09-20 01:34:30,2026-09-20 01:44:49,NYPD,New York City Police Department,Noise - Commercial,Banging/Pounding,NaN,Club/Bar/Restaurant,10002,...,NaN,NaN,NaN,NaN,NaN,NaN,40.718013,-73.989896,POINT (-73.989895597843 40.718012546099),0.171944


### Select feature columns

In [122]:
selected_columns = [
    'Created Date',
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Agency',
    'Borough',
    'Location Type',
    'Open Data Channel Type',
    'Community Board',
    'Police Precinct',
    'resolution_hours',
]

df = df[selected_columns]
print(f"Shape: {df.shape}")
df.head()

Shape: (3759919, 10)


,Created Date,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Agency,Borough,Location Type,Open Data Channel Type,Community Board,Police Precinct,resolution_hours
75,2026-09-20 01:39:36,Noise - Residential,Loud Music/Party,NYPD,BROOKLYN,Residential Building/House,ONLINE,05 BROOKLYN,Precinct 75,0.132222
90,2026-09-20 01:37:27,Noise - Street/Sidewalk,Loud Music/Party,NYPD,BROOKLYN,Street/Sidewalk,MOBILE,01 BROOKLYN,Precinct 94,0.166389
95,2026-09-20 01:36:25,Noise - Park,Loud Music/Party,NYPD,MANHATTAN,Park/Playground,ONLINE,12 MANHATTAN,Precinct 33,0.149444
99,2026-09-20 01:35:40,Noise - Street/Sidewalk,Loud Talking,NYPD,BROOKLYN,Street/Sidewalk,MOBILE,08 BROOKLYN,Precinct 77,0.230278
106,2026-09-20 01:34:30,Noise - Commercial,Banging/Pounding,NYPD,MANHATTAN,Club/Bar/Restaurant,PHONE,03 MANHATTAN,Precinct 7,0.171944


### Missing values check

In [123]:
missing = df.isna().sum().sort_values(ascending=False)
print (missing)

Location Type                           463792
Problem Detail (formerly Descriptor)     73601
Created Date                                 0
Problem (formerly Complaint Type)            0
Agency                                       0
Borough                                      0
Open Data Channel Type                       0
Community Board                              0
Police Precinct                              0
resolution_hours                             0
dtype: int64


`Location Type` column has 463,792 missing values and `Problem Detail` has 73,601 missing values-> too many to drop -> fill it with `Not Specified`

In [124]:
df['Location Type'] = df['Location Type'].fillna('Not Specified')
df['Problem Detail (formerly Descriptor)'] = df['Problem Detail (formerly Descriptor)'].fillna('Not Specified')
missing = df.isna().sum().sort_values(ascending=False)
print (missing)

Created Date                            0
Problem (formerly Complaint Type)       0
Problem Detail (formerly Descriptor)    0
Agency                                  0
Borough                                 0
Location Type                           0
Open Data Channel Type                  0
Community Board                         0
Police Precinct                         0
resolution_hours                        0
dtype: int64


### Standardize categorical casing

Switched to a simpler approach: lowercase every categorical column entirely, rather than only fixing the ALL CAPS values. This avoids the apostrophe-mangling issue `.title()` has (`"Doctor's Office"` → `"Doctor'S Office"`) and guarantees casing can never cause a false "different category" again, for any column, not just the two checked so far.

In [125]:
categorical_cols = df.select_dtypes(include='object').columns
before_n = {col: df[col].nunique() for col in categorical_cols}

for col in categorical_cols:
    df[col] = df[col].str.lower()

for col in categorical_cols:
    print(f"{col}: {before_n[col]} -> {df[col].nunique()} unique values")

Problem (formerly Complaint Type): 191 -> 189 unique values
Problem Detail (formerly Descriptor): 1009 -> 1006 unique values
Agency: 16 -> 16 unique values
Borough: 6 -> 6 unique values
Location Type: 153 -> 152 unique values
Open Data Channel Type: 5 -> 5 unique values
Community Board: 77 -> 77 unique values
Police Precinct: 78 -> 78 unique values


The number of the unique values is dropped but not significantly -> there is still an issue with the naming

#### Complaint Type

<mark>Noise</mark>

In [126]:
noise = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
noise[noise.index.astype(str).str.contains('noise', case=False, na=False)]

Problem (formerly Complaint Type)
noise - residential         433900
noise - street/sidewalk     175293
noise - commercial           60070
noise                        58140
noise - vehicle              51040
noise - park                  8460
noise - house of worship      2920
noise - helicopter             181
Name: count, dtype: int64

In [127]:
col = 'Problem (formerly Complaint Type)'
noise_df = df[df[col].str.contains('noise', case=False, na=False)]
noise_summary = noise_df.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
noise_summary

,count,median_hours
Problem (formerly Complaint Type),,
noise - house of worship,2920,0.450000
noise - commercial,60070,0.965556
noise - vehicle,51040,0.967778
noise - park,8460,1.017778
noise - street/sidewalk,175293,1.118611
noise - residential,433900,1.173611
noise,58140,68.950000
noise - helicopter,181,6550.935278


In [128]:
def merge_categories(df, column, values_to_merge, new_value):
    df[column] = df[column].replace({v: new_value for v in values_to_merge})
    return df

In [129]:
noise_to_merge = [
    'noise - house of worship',
    'noise - commercial',
    'noise - vehicle',
    'noise - park',
    'noise - street/sidewalk',
    'noise - residential',
]

df = merge_categories(df, 'Problem (formerly Complaint Type)', noise_to_merge, 'noise - general')

***Findings:*** median resolution time of `residential`, `street/sidewalk`, `commercial`, `vehicle`, `park`, `house of worship` all clustered tightly between 0.45–1.17 hours, indicating they're handled the same way in practice —> can be merged 

Generic `noise` (69h) and `noise - helicopter` (~9 months) are kept separately

<mark>Sign</mark>

In [130]:
sign = df['Problem (formerly Complaint Type)'].value_counts()
sign[sign.index.astype(str).str.contains('sign', case=False, na=False)]

Problem (formerly Complaint Type)
traffic signal condition    56922
street sign - damaged        5082
street sign - dangling       3870
street sign - missing        3076
highway sign - damaged         73
highway sign - missing         39
highway sign - dangling        19
Name: count, dtype: int64

In [131]:
col = 'Problem (formerly Complaint Type)'
sign_df = df[df[col].str.contains('sign', case=False, na=False)]
sign_summary = sign_df.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
sign_summary

,count,median_hours
Problem (formerly Complaint Type),,
traffic signal condition,56922,2.350000
street sign - missing,3076,100.046528
street sign - damaged,5082,111.516250
highway sign - dangling,19,133.852778
street sign - dangling,3870,141.794583
highway sign - damaged,73,145.088889
highway sign - missing,39,227.011667


In [132]:
df = merge_categories(
    df, col,
    ['street sign - missing', 'street sign - damaged', 'street sign - dangling'],
    'street sign',
)

df = merge_categories(
    df, col,
    ['highway sign - missing', 'highway sign - damaged', 'highway sign - dangling'],
    'highway sign',
)

***Findings:*** Two more genuine families turned up: `street sign` and `highway sign` - both homogeneous enough (no extreme outlier) to merge without excluding anything.

<mark>Marshal</mark>

In [133]:
marshal = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
marshal[marshal.index.astype(str).str.contains('marshal', case=False, na=False)]

Problem (formerly Complaint Type)
building marshal's office    18
building marshals office      1
Name: count, dtype: int64

In [134]:
sub = df[df[col].isin(["building marshal's office", 'building marshals office'])]
summary = sub.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                                   count  median_hours
Problem (formerly Complaint Type)                     
building marshals office               1    230.182222
building marshal's office             18    336.049028


In [135]:
df = merge_categories(df, col, ["building marshal's office", 'building marshals office'], "building marshal's office")

<mark>Hazardous</mark>

In [136]:
hazardous = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
hazardous[hazardous.index.astype(str).str.contains('hazardous', case=False, na=False)]

Problem (formerly Complaint Type)
hazardous materials    1743
hazardous material      424
Name: count, dtype: int64

In [137]:
sub = df[df[col].isin(['hazardous material', 'hazardous materials'])]
summary = sub.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                                   count  median_hours
Problem (formerly Complaint Type)                     
hazardous materials                 1743      2.500000
hazardous material                   424      2.520556


In [138]:
df = merge_categories(df, col, ['hazardous material', 'hazardous materials'], 'hazardous materials')

<mark>Electric VS Electrical</mark>

In [139]:
electric = df['Problem (formerly Complaint Type)'].value_counts(dropna=False)
electric[electric.index.astype(str).str.contains('electric', case=False, na=False)]

Problem (formerly Complaint Type)
electric      32986
electrical     2481
Name: count, dtype: int64

In [140]:
sub = df[df[col].isin(['electric', 'electrical'])]
summary = sub.groupby(col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                                   count  median_hours
Problem (formerly Complaint Type)                     
electrical                          2481    103.981944
electric                           32986    181.963750


In [141]:
agency_counts = pd.crosstab(sub[col], sub['Agency'])
print(agency_counts)

Agency                              dob    hpd
Problem (formerly Complaint Type)             
electric                              0  32986
electrical                         2481      0


In [142]:
df = merge_categories(df, col, ['electric', 'electrical'], 'electrical')

***Findings:*** `electric` is 100% HPD, `electrical` is 100% DOB -> different agency, not a different real category

In [143]:
#Improval Check
for col in categorical_cols:
    print(f"{col}: {before_n[col]} -> {df[col].nunique()} unique values")

Problem (formerly Complaint Type): 191 -> 177 unique values
Problem Detail (formerly Descriptor): 1009 -> 1006 unique values
Agency: 16 -> 16 unique values
Borough: 6 -> 6 unique values
Location Type: 153 -> 152 unique values
Open Data Channel Type: 5 -> 5 unique values
Community Board: 77 -> 77 unique values
Police Precinct: 78 -> 78 unique values


#### Location Type

<mark>Residential Building</mark>

In [144]:
loc_col = 'Location Type'
residential = df[loc_col].value_counts(dropna=False)
residential[residential.index.astype(str).str.contains('residential', case=False, na=False)]

Location Type
residential building          857620
residential building/house    473922
residential                     3579
building (non-residential)      1602
residential property             470
Name: count, dtype: int64

In [145]:
sub = df[df[loc_col].isin(['residential building/house', 'residential building'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                             count  median_hours
Location Type                                   
residential building/house  473922      1.210000
residential building        857620     93.522778


In [146]:
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

Agency                      dep   dhs  dohmh   hpd  nypd
Location Type                                           
residential building        0.0  0.00   0.01  0.99  0.00
residential building/house  0.0  0.02   0.00  0.00  0.98


In [147]:
df = merge_categories(df, loc_col, ['residential building/house', 'residential building'], 'residential building')

***Findings:*** `residential building/house` is 98% NYPD, `residential building` is 99% HPD -> different agency, not a different real category

<mark>Comercial vs Commercial</mark>

In [148]:
comercial = df[loc_col].value_counts(dropna=False)
comercial[comercial.index.astype(str).str.contains('comercial|commercial', case=False, na=False)]

Location Type
store/commercial                 54433
comercial                         4221
commercial building               3347
commercial/mixed use building     1293
commercial                          75
commercial property                 67
Name: count, dtype: int64

In [149]:
sub = df[df[loc_col].isin(['comercial', 'commercial'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

               count  median_hours
Location Type                     
commercial        75      1.928333
comercial       4221    177.546111
Agency         dsny  nypd
Location Type            
comercial       1.0   0.0
commercial      0.0   1.0


In [150]:
df = merge_categories(df, loc_col, ['comercial', 'commercial'], 'commercial')

***Findings:*** `comercial` is 100% DSNY (a typo baked into their intake system), `commercial` is 100% NYPD -> different agency, not a different real category

<mark>Catch Basin</mark>

In [151]:
catch_basin = df[loc_col].value_counts(dropna=False)
catch_basin[catch_basin.index.astype(str).str.contains('catch basin', case=False, na=False)]

Location Type
catch basin/sewer       205
catch basin or sewer    139
Name: count, dtype: int64

In [152]:
sub = df[df[loc_col].isin(['catch basin or sewer', 'catch basin/sewer'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

                      count  median_hours
Location Type                            
catch basin/sewer       205      9.282500
catch basin or sewer    139    170.540556
Agency                dep  dohmh
Location Type                   
catch basin or sewer  0.0    1.0
catch basin/sewer     1.0    0.0


In [153]:
df = merge_categories(df, loc_col, ['catch basin or sewer', 'catch basin/sewer'], 'catch basin/sewer')

***Findings:*** `catch basin or sewer` is 100% DOHMH, `catch basin/sewer` is 100% DEP -> different agency, not a different real category

<mark>Mixed Use</mark>

In [154]:
mixed_use = df[loc_col].value_counts(dropna=False)
mixed_use[mixed_use.index.astype(str).str.contains('mixed use', case=False, na=False)]

Location Type
mixed use                        6136
3+ family mixed use building     2519
commercial/mixed use building    1293
1-2 family mixed use building     890
1-3 family mixed use building      10
mixed use building                  8
Name: count, dtype: int64

In [155]:
sub = df[df[loc_col].isin(['mixed use building', 'mixed use'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

                    count  median_hours
Location Type                          
mixed use building      8     48.326111
mixed use            6136    155.650694
Agency              dohmh  dsny
Location Type                  
mixed use             0.0   1.0
mixed use building    1.0   0.0


In [156]:
df = merge_categories(df, loc_col, ['mixed use building', 'mixed use'], 'mixed use')

***Findings:*** `mixed use building` is 100% DOHMH, `mixed use` is 100% DSNY -> different agency, not a different real category. The other `mixed use` variants (`3+ family mixed use building`, `1-2 family mixed use building`, etc.) are genuinely distinct family-count categories, not duplicates - left separate.

<mark>Parking Lot</mark>

In [157]:
parking_lot = df[loc_col].value_counts(dropna=False)
parking_lot[parking_lot.index.astype(str).str.contains('parking lot', case=False, na=False)]

Location Type
parking lot or garage    301
parking lot               89
parking lot/garage         4
Name: count, dtype: int64

In [158]:
sub = df[df[loc_col].isin(['parking lot/garage', 'parking lot or garage'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

                       count  median_hours
Location Type                             
parking lot/garage         4      71.64875
parking lot or garage    301      72.32000
Agency                 dohmh
Location Type               
parking lot or garage    1.0
parking lot/garage       1.0


In [159]:
df = merge_categories(df, loc_col, ['parking lot/garage', 'parking lot or garage'], 'parking lot or garage')

***Findings:*** both 100% DOHMH - not an agency split, but medians are already near-identical (71.6h vs 72.3h), so merging is safe regardless

<mark>Subway</mark>

In [160]:
subway = df[loc_col].value_counts(dropna=False)
subway[subway.index.astype(str).str.contains('subway', case=False, na=False)]

Location Type
subway            11088
subway station      120
Name: count, dtype: int64

In [161]:
sub = df[df[loc_col].isin(['subway station', 'subway'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

                count  median_hours
Location Type                      
subway          11088      0.877361
subway station    120      1.033889
Agency           dhs  nypd
Location Type             
subway          0.42  0.58
subway station  0.00  1.00


In [162]:
df = merge_categories(df, loc_col, ['subway station', 'subway'], 'subway')

***Findings:*** `subway station` is 100% NYPD, `subway` is a 58%/42% NYPD/DHS split - not a clean agency divide, but medians are already close (0.88h vs 1.03h), so merging is safe regardless

<mark>Other</mark>

In [163]:
other = df[loc_col].value_counts(dropna=False)
other[other.index.astype(str).str.contains('other', case=False, na=False)]

Location Type
other                                   5442
other (explain below)                   2266
other (explain in complaint details)      46
Name: count, dtype: int64

In [164]:
other_pair = ['other (explain below)', 'other (explain in complaint details)']
sub = df[df[loc_col].isin(['other'] + other_pair)]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)
print(pd.crosstab(sub[loc_col], sub['Agency'], normalize='index').round(2))

                                      count  median_hours
Location Type                                            
other                                  5442      4.726389
other (explain below)                  2266   1226.681667
other (explain in complaint details)     46   1440.165972
Agency                                 dep   dhs  dohmh  nypd
Location Type                                                
other                                 0.02  0.61   0.15  0.23
other (explain below)                 0.00  0.00   1.00  0.00
other (explain in complaint details)  0.00  0.00   1.00  0.00


In [165]:
df = merge_categories(df, loc_col, other_pair, 'other (explain further)')

***Findings:*** `other (explain below)` and `other (explain in complaint details)` are both 100% DOHMH and reasonably close (1,227h vs 1,440h) -> merged with each other. Generic `other` is spread across DHS/DOHMH/NYPD and behaves completely differently (4.7h) -> kept separate, merging it in would blend a fast multi-agency catch-all with a slow DOHMH-specific one

<mark>3+ Family Apartment Building</mark>

In [166]:
family = df[loc_col].value_counts(dropna=False)
family[family.index.astype(str).str.contains('3+ family', case=False, na=False, regex=False)]

Location Type
3+ family apt. building         12942
3+ family apartment building     9201
3+ family mixed use building     2519
Name: count, dtype: int64

In [167]:
family_pair = ['3+ family apt. building', '3+ family apartment building']
sub = df[df[loc_col].isin(family_pair)]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                              count  median_hours
Location Type                                    
3+ family apartment building   9201     31.482778
3+ family apt. building       12942     58.813750


In [168]:
df = merge_categories(df, loc_col, family_pair, '3+ family apartment building')

***Findings:*** both 100% DOHMH - the 1.9x gap isn't agency-explained, but far more modest than the other gaps found here, and these are clearly the same category, just abbreviated -> merged

<mark>Public Garden</mark>

In [169]:
garden = df[loc_col].value_counts(dropna=False)
garden[garden.index.astype(str).str.contains('garden', case=False, na=False)]

Location Type
public park/garden    131
public garden          96
public garden/park      8
Name: count, dtype: int64

In [170]:
sub = df[df[loc_col].isin(['public garden', 'public park/garden'])]
summary = sub.groupby(loc_col)['resolution_hours'].agg(count='count', median_hours='median').sort_values('median_hours')
print(summary)

                    count  median_hours
Location Type                          
public park/garden    131      0.321667
public garden          96     66.268333


***Findings:*** both `public garden` and `public park/garden` are 100% DOHMH - same agency, yet a 206x time gap with no explanation found. Left separate rather than guessed at.

In [171]:
#Improval Check
for col in categorical_cols:
    print(f"{col}: {before_n[col]} -> {df[col].nunique()} unique values")

Problem (formerly Complaint Type): 191 -> 177 unique values
Problem Detail (formerly Descriptor): 1009 -> 1006 unique values
Agency: 16 -> 16 unique values
Borough: 6 -> 6 unique values
Location Type: 153 -> 144 unique values
Open Data Channel Type: 5 -> 5 unique values
Community Board: 77 -> 77 unique values
Police Precinct: 78 -> 78 unique values


### Consolidate rare categories into "Other"

Checked the actual long tail before picking a threshold: a category with only a handful of rows can't teach a model much, but bucketing too aggressively throws away real information. A 0.1% of total rows cutoff (~3,900 rows) drops the truly rare categories in `Problem (formerly Complaint Type)` and `Location Type` while affecting only ~1-2% of total data — a big reduction in category count for very little information loss.

In [172]:
def consolidate_rare_categories(df, column, threshold=0.001, other_label='other'):
    counts = df[column].value_counts()
    rare_values = counts[counts < len(df) * threshold].index
    before_n = df[column].nunique()
    df[column] = df[column].where(~df[column].isin(rare_values), other_label)
    print(f'{column}: {before_n} -> {df[column].nunique()} unique values ({len(rare_values)} bucketed into "{other_label}")')
    return df

df = consolidate_rare_categories(df, 'Problem (formerly Complaint Type)')
df = consolidate_rare_categories(df, 'Location Type')

Problem (formerly Complaint Type): 177 -> 73 unique values (105 bucketed into "other")
Location Type: 144 -> 22 unique values (122 bucketed into "other")
